# Modules

In [1]:
# Import required libraries
from Pfitting_utility import *
import pickle
from natsort import natsorted

In [2]:
%load_ext autoreload
%autoreload 2

# Data

In [3]:
### Get data -- case number is arbitrary (just labelled as used)

case_i = 'case1'
ddir = 'Greebo_profiles/f444w/L3_PA20.0'

# case_i = 'case2'
# ddir = 'Greebo_profiles/f444w/L1_PA6.0'

# case_i = 'case3'
# ddir = 'Greebo_profiles/f444w/L2_PA6.0'

df_files = natsorted(glob.glob(f"{ddir}/tables2/df*"))
gp_files = natsorted(glob.glob(f"{ddir}/tables2/gp*"))

data_ls = [pd.read_csv(df_dir) for df_dir in df_files]
gp_ls = [pd.read_csv(gp_dir) for gp_dir in gp_files]

In [4]:
### for plotting
colorblind_colors = ['#0173B2', '#DE8F05', '#CC78BC', '#CA9161', '#949494','#ECE133', '#56B4E9', '#009E73', '#F0E442', '#D55E00','#E69F00', '#029E73', '#56B4E9', '#CC79A7', '#999999']
recommended_colors = [
    '#1f77b4',  # Muted blue
    '#ff7f0e',  # Safety orange
    '#2ca02c',  # Cooked asparagus green
    '#d62728',  # Brick red
    '#9467bd',  # Muted purple
    '#8c564b',  # Chestnut brown
    '#e377c2',  # Raspberry yogurt pink
    '#7f7f7f',  # Middle gray
    '#bcbd22',  # Curry yellow-green
    '#17becf',  # Blue-teal
    '#aec7e8',  # Light blue
    '#ffbb78',  # Light orange
    '#98df8a',  # Light green
    '#ff9896',  # Light red
    '#c5b0d5'   # Light purple
]

In [5]:
### Model example
### Using piece wise exponential functional form to describe light curve

# t_range = np.linspace(-20,20,1000)
# func_t = piece_wise(t_range, 0.0, 0.5, 2.0)

# plt.plot(t_range, func_t)
# rt, dt = get_hwhm(t_range, func_t, plot=False)
# print(rt, dt)

# Fitting Process

## Testing emcee code

In [37]:
# test_r, test_d = 0.1, 1.0
# # test_r, test_d = None, None
# test_nuis = False
# # test_nuis = True
# test = emcee_fit(fit_data, test_r, test_d, fit4nuisance=test_nuis)

In [38]:
# plot_emcee(test, fit_data, test_r, test_d, fit4nuisance=test_nuis)

In [39]:
# if test_r is not None:
#     if test_nuis is False:
#         labels_t = ['sigma','t0']
#     elif test_nuis is True:
#         labels_t = ['sigma','t0', 'A', 'B']
# elif test_r is None:
#     if test_nuis is False:
#         labels_t = ['sigma','t0', 'hwhm_rise', 'hwhm_delta']
#     elif test_nuis is True:
#         labels_t = ['sigma','t0', 'hwhm_rise', 'hwhm_delta', 'A', 'B']

# corner.corner(test, labels=labels_t)

## Emcee fitting - Grid Approach

In [6]:
### Select image to do peak fitting on LE

if case_i == 'case1': sel_ind = 4
if case_i == 'case2': sel_ind = 3
if case_i == 'case3': sel_ind = 5
    
fit_data = data_ls[sel_ind]

In [7]:
### Define grid of rise and fall times to run fitting process

fit_trises = [1./24., 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
fit_tfalls = []
fit_tdeltas = np.arange(0.05, 7.25, 0.25)

for ftr in fit_trises:
    temp_tfalls = ftr+fit_tdeltas
    fit_tfalls.append(temp_tfalls)

In [40]:
### Run through process and save fitting results -- takes a lot of memory so may crash so rerun

# # em_sampss = []

# for ftr in fit_trises:
#     # em_samps = []
#     for ftd in fit_tdeltas:
#         # em_samp = emcee_fit(fit_data, ftr, ftd)
#         # em_samps.append(em_samp)
#     # em_sampss.append(em_samps)
#         if os.path.isfile(f'{ddir}/fitting_params/grid/run_{ftr}_{ftd}.pkl') is False:     # can't run over all grid at once, so skip ones that did run
#             em_samp = emcee_fit(fit_data, ftr, ftd)
#             with open(f'{ddir}/fitting_params/grid/run_{ftr}_{ftd}.pkl', 'wb') as f:
#                 pickle.dump(em_samp, f)

In [41]:
### Read in fitting results

em_sampss = []
for ftr in fit_trises:
    em_samps = []
    for ftd in fit_tdeltas:
        with open(f'{ddir}/fitting_params/grid/run_{ftr}_{ftd}.pkl', 'rb') as f:
            em_samp = pickle.load(f)
            em_samps.append(em_samp)
    em_sampss.append(em_samps)

In [ ]:
### Plot fitting results

# for i in range(len(fit_trises)):
#     one_fig, one_axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
#     one_axs = one_axs.flatten()
#     for j, em in enumerate(em_sampss[0]):
#         # if j==0 or j==8 or j==15 or j==25:
#         if j==0 or j==25:
#             plot_emcee3(em, fit_data, fit_trises[i], fit_tdeltas[j], figure=[one_fig,one_axs], color_q=recommended_colors[j//4])
#     plt.show()

In [ ]:
### Get reduced chi^2 of the fits

# chi2_lss = []
# for i, ems in enumerate(em_sampss):
#     chi2_ls = []
#     for j, em in enumerate(ems):
#         sigma_res, t0_res, chi2_res, resids = emcee3_results(em, fit_data, fit_trises[i], fit_tdeltas[j])
#         chi2_ls.append(chi2_res)
#     chi2_lss.append(chi2_ls)

In [ ]:
### Calculate the p-values of the chi^2's

# dof_i = len(fit_data) - 2

# p_valss = []
# p_cutoffs = []
# for i, chi2s in enumerate(chi2_lss):
#     p_vals = calc_pvalchi2(chi2s, dof_i, fit_tfalls[i])
#     p_valss.append(p_vals)

#     pval_f = interp1d(fit_tfalls[i], p_vals)
#     proot = find_root(pval_f, 0.05, [min(fit_tfalls[i]),max(fit_tfalls[i])])
#     p_cutoffs.append(proot)

In [ ]:
### Look at loglikelihood distribution

# for i, ftr in enumerate(fit_trises):
#     chi2_i = chi2_lss[i]
#     llike = -0.5*np.array(chi2_i)
#     llike_norm = np.exp(llike-llike.max())
#     plt.scatter(fit_tfalls[i], llike_norm)
    
#     ll_cut = 0.75
#     ll_f = interp1d(fit_tfalls[i], llike_norm)
#     x_cut = opt.fsolve(lambda x: ll_f(x) - ll_cut, x0=5.5)[0]
#     plt.axhline(ll_cut, color='k')
#     plt.axvline(x_cut, color='k')
    
#     plt.title(f'Rise time of {ftr:.2f}')
#     plt.xlabel('Fall time (days)')
#     plt.ylabel('Normalized Likelihood')
#     plt.show()

In [ ]:
### Plot chi2's on grid of rise and fall times

# max_tdur = fit_trises[-1] + fit_tfalls[-1][-1]

# dof_i = len(fit_data) - 2
# chi2_16th = sc_chi2.ppf(0.16, dof_i)/dof_i
# chi2_50th = sc_chi2.ppf(0.5, dof_i)/dof_i
# chi2_90th = sc_chi2.ppf(0.9, dof_i)/dof_i

# colorst = ['red', 'blue', 'green']

# import matplotlib.colors as mcolors
# norm = mcolors.Normalize(vmin=1.0, vmax=2.0)

# for i in range(len(em_sampss)):
#     plt.scatter([fit_trises[i]]*len(fit_tfalls[i]), fit_tfalls[i], c=chi2_lss[i], norm=norm)

# plt.axline((0,0),slope=1.0, color='k', ls='--', alpha=0.5)
# plt.colorbar(label='$\chi^2$')

# plt.xlabel('Rise time (days)')
# plt.ylabel('Fall time (days)')
# # plt.ylabel('Delta (days)')

# plt.savefig(f"plots/4update/{case_i}/fallvrise_chi2grid(1).png", bbox_inches = 'tight', dpi=100)

In [ ]:
### Plot p-values on grid of rise and fall times

# norm = mcolors.Normalize(vmin=0.0, vmax=1.0)

# for i in range(len(em_sampss)):
#     plt.scatter([fit_trises[i]]*len(fit_tfalls[i]), fit_tfalls[i], c=p_valss[i], norm=norm, cmap='viridis_r')

# plt.axline((0,0),slope=1.0, color='k', ls='--', alpha=0.5)

# plt.colorbar(label='$p-value$')

# plt.xlabel('Rise time (days)')
# plt.ylabel('Fall time (days)')
# # plt.ylabel('Delta (days)')

# plt.legend()

# plt.savefig(f"plots/4update/{case_i}/fallvrise_pvalgrid(1).png", bbox_inches = 'tight', dpi=100)

In [ ]:
### Plot chi2's per rise time

# n_plots = len(em_sampss)
# fig, axs = plt.subplots(n_plots//3+1, 3, figsize=(12, n_plots//3*4))
# axs = axs.flatten()
# for i in range(len(em_sampss)):
#     ax = axs[i]
#     ax.plot(fit_tfalls[i], chi2_lss[i], ls='-', marker='o', color='royalblue')
#     ax.set_ylabel('Reduced $\\chi^2$', fontsize=10)
#     ax.set_xlabel('Fall Time (days)', fontsize=10)
#     ax.set_title(f"Rise Time = {fit_trises[i]:.2f} days", fontsize=12)
#     ax.axhline(min(chi2_lss[i])+chi2_90th, color='red')
#     chi2_f = interp1d(fit_tfalls[i], chi2_lss[i])

#     cutoff = min(chi2_lss[i]) + chi2_90th
#     root = find_root(chi2_f, cutoff, [min(fit_tfalls[i]), max(fit_tfalls[i])])
#     ax.axvline(root, color='red', label=f"{root:.2f} days")
            
#     ax.legend(loc=2)
# fig.subplots_adjust(wspace=0.25, hspace=0.4)

# fig.savefig(f'plots/4update/{case_i}/fallvrise_chi2s(1).png', bbox_inches = 'tight', dpi=100)

In [ ]:
### Better visualize comparison of fall times

# one_fig, one_axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
# one_axs = one_axs.flatten()
# for j, em in enumerate(em_sampss[0]):
#     if j==0 or j==8 or j==15 or j==25:
#         plot_emcee3(em, fit_data, fit_trises[0], fit_tdeltas[j], figure=[one_fig,one_axs], color_q=recommended_colors[j//4])

# one_fig.savefig(f"plots/4update/{case_i}/fits_plot(1).png", bbox_inches = 'tight', dpi=100)

In [ ]:
### Check corner plots of fits

# for i, ems in enumerate(em_sampss):
#     if i == 0:
#         for j, em in enumerate(ems):
#             corner.corner(em, labels=['sigma','t0'], quantiles=[0.16, 0.5, 0.84], levels=[0.68, 0.95])
#             plt.show()

## Dynesty Approach

### Check all epochs

In [ ]:
# dyn_results = []
# for i, fdata in enumerate(data_ls):
#     dyn_res = dynesty_fit(fdata)
#     dyn_results.append(dyn_res)

    # with open(f'{ddir}/fitting_params/dynesty/dynresults_epoch{i+1}.pkl', 'wb') as f:
    #     pickle.dump(dyn_res, f)

In [ ]:
## read in previously saved runs

# dyn_files = natsorted(glob.glob(f'{ddir}/fitting_params/dynesty/*.pkl'))
# dyn_results = []
# for fi in dyn_files:
#     with open(fi, 'rb') as f:
#         dyn_res = pickle.load(f)
#         dyn_results.append(dyn_res)

In [ ]:
# for i, dynres in enumerate(dyn_results):
#     print(i)
#     print(f"------------------")
#     fdata = data_ls[i]
#     fig_fit, ax_fit = plot_dynesty(dynres, fdata)
#     plt.show()

In [ ]:
# for i, dynres in enumerate(dyn_results):
#     print(i)
#     print(f"------------------")
#     fig = dyplot.cornerplot(dynres, labels=['hwhm_trise','hwhm_delta', 'sigma', 't0'], title_quantiles=[0.16, 0.5, 0.84], quantiles_2d=[0.68, 0.95, 0.997])
#     plt.show()

### Check Best Epoch

In [ ]:
# if case_i == 'case1': sel_ind = 4
# if case_i == 'case2': sel_ind = 3
# if case_i == 'case3': sel_ind = 5
    
# with open(f'{ddir}/fitting_params/dynesty/dynresults_epoch{sel_ind+1}.pkl', 'rb') as f:
#     dynres_c= pickle.load(f)

# fit_data = data_ls[sel_ind]

In [ ]:
# weights = np.exp(dynres_c.logwt - dynres_c.logz[-1])
# weights /= sum(weights)
# samples_equal = dynesty.utils.resample_equal(dynres_c.samples, weights)

In [ ]:
# fig_fit, ax_fit = plot_dynesty(dynres_c, fit_data)
# fig_fit.savefig(f'plots/4update/{case_i}/dynesty_fit.png', bbox_inches = 'tight', dpi=100)

In [ ]:
# fig = dyplot.cornerplot(dynres_c, labels=['hwhm_trise','hwhm_delta', 'sigma', 't0'], title_quantiles=[0.16, 0.5, 0.84], quantiles_2d=[0.68, 0.95, 0.997])
# fig[0].savefig(f'plots/4update/{case_i}/dynesty_res.png', bbox_inches = 'tight', dpi=100)

## Simultaneous Multiple-Peak Approach

In [ ]:
# simul_fit = simul_emcee(data_ls[3:6])

In [ ]:
# plot_simul(simul_fit, data_ls[3:6])

In [ ]:
# corner.corner(simul_fit, labels=['hwhm_rise', 'hwhm_delta', 'sigma', 't0'], show_titles=True, title_fmt=".3f")